## Expanding Window Portfolio Transformer

Gu, Kelly, and Xiu (2020) expanding window protocol. The model retrains once per year on all available data up to that point. The last 24 months of each expanding window serve as validation for early stopping. Predictions cover the next 12 months out of sample. Self contained: loads raw JKP parquet, processes, trains, and evaluates.

### 1. Setup

In [ ]:
import gc
import json
import pickle
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.size'] = 11

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}, device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

### 2. Configuration

In [ ]:
country = 'IND'
raw_path = Path('../data/Global Factor_IND.parquet')
results_dir = Path('../results') / country / 'expanding'
results_dir.mkdir(parents = True, exist_ok = True)

# Data processing
coverage_threshold = 0.70
max_miss_frac = 1.0 / 3.0
min_stocks = 30

# Expanding window
initial_train_years = 10
val_size_months = 24

# Architecture (set from Optuna tuning results)
n_blocks = 2
n_heads = 1
d_ff = 256

# Training
n_epochs = 50
lr = 1e-5
weight_decay = 1e-3
grad_clip = 1.0
n_seeds = 3
patience = 10

# Portfolio
rebalance_freq = 6
tc_bps = 25

# Columns
load_always = ['id', 'gvkey', 'eom', 'excntry', 'ret_exc_lead1m', 'me']
exclude_cols = {
	'id', 'gvkey', 'iid', 'permno', 'permco', 'date', 'eom', 'excntry',
	'size_grp', 'obs_main', 'exch_main', 'common', 'primary_sec',
	'source_crsp', 'comp_tpci', 'crsp_shrcd', 'comp_exchg', 'crsp_exchcd',
	'curcd', 'fx', 'adjfct', 'bidask',
	'ret', 'ret_local', 'ret_exc', 'ret_exc_lead1m',
	'prc', 'prc_local', 'prc_high', 'prc_low',
	'me', 'me_company', 'dolvol', 'shares', 'tvol',
	'ret_lag_dif', 'div_tot',
}

variant_list = ['identity', 'linear', 'ple', 'periodic', 'fourier', 'magnitude_dir']

print(f'country: {country}')
print(f'initial training: {initial_train_years} years, val: {val_size_months} months')

### 3. Data Processing

In [ ]:
schema = pq.read_schema(raw_path)
all_col_names = schema.names
char_candidate = [c for c in all_col_names if c not in exclude_cols and c not in load_always]
needed = [c for c in load_always + char_candidate if c in all_col_names]
df = pd.read_parquet(raw_path, columns = needed)
df['eom'] = pd.to_datetime(df['eom'])
print(f'loaded: {df.shape[0]:,} rows, {df["eom"].min().date()} to {df["eom"].max().date()}')

for col in char_candidate:
	if col in df.columns and df[col].dtype == np.float64: df[col] = df[col].astype(np.float32)
char_candidate = [c for c in char_candidate if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

# Coverage on initial training period
sorted_eoms = sorted(df['eom'].unique())
initial_cutoff = sorted_eoms[min(initial_train_years * 12 - 1, len(sorted_eoms) - 1)]
df_cov = df[df['eom'] <= initial_cutoff]
coverage = df_cov[char_candidate].notna().mean()
char_cols = sorted([c for c in char_candidate if coverage[c] >= coverage_threshold])
d = len(char_cols)
print(f'd = {d} features (coverage on {initial_cutoff.date()})')
del df_cov

# Firm lookup
info_cols = [c for c in ['id', 'gvkey', 'excntry'] if c in df.columns]
firm_lookup = df[info_cols].drop_duplicates(subset = ['id'])
if 'excntry' not in firm_lookup.columns: firm_lookup['excntry'] = country
id_to_gvkey = dict(zip(firm_lookup['id'], firm_lookup.get('gvkey', firm_lookup['id'])))

# Process each month
id_cols = [c for c in load_always if c in df.columns]
df = df[id_cols + char_cols]
n_miss = df[char_cols].isna().sum(axis = 1)
df = df[n_miss <= d * max_miss_frac].reset_index(drop = True)

all_months = {}
for eom in sorted_eoms:
	month = df[df['eom'] == eom].copy()
	if len(month) < min_stocks: continue
	ranks = month[char_cols].rank(pct = True, axis = 0) - 0.5
	month[char_cols] = ranks.fillna(0.0)
	x = month[char_cols].values.astype(np.float32)
	r = month['ret_exc_lead1m'].values.astype(np.float32)
	ids = month['id'].values
	hr = np.isfinite(r)
	if hr.sum() >= 5:
		all_months[eom] = {'x': x[hr], 'r': r[hr], 'ids': ids[hr]}

del df
gc.collect()
sorted_dates = sorted(all_months.keys())
n_total = len(sorted_dates)

# Determine annual boundaries
years = sorted(set(d.year for d in sorted_dates))
first_oos_year = years[0] + initial_train_years
oos_years = [y for y in years if y >= first_oos_year]
n_oos_months = sum(1 for d in sorted_dates if d.year >= first_oos_year)

print(f'processed: {n_total} months, ~{np.mean([m["x"].shape[0] for m in all_months.values()]):.0f} firms')
print(f'OOS years: {oos_years[0]} to {oos_years[-1]} ({len(oos_years)} retrains, ~{n_oos_months} OOS months)')

### 4. Encoding Variants

In [ ]:
class IdentityEncoder(nn.Module):
	def forward(self, x): return x

class LinearEncoder(nn.Module):
	def __init__(self, n):
		super().__init__()
		self.w = nn.Parameter(torch.ones(n))
		self.b = nn.Parameter(torch.zeros(n))
	def forward(self, x): return x * self.w + self.b

class PLEEncoder(nn.Module):
	def __init__(self, n, bins = 16):
		super().__init__()
		bd = torch.linspace(-0.5, 0.5, bins + 1)
		self.register_buffer('lo', bd[:-1])
		self.register_buffer('hi', bd[1:])
		self.w = nn.Parameter(torch.zeros(n, bins))
	def forward(self, x):
		a = torch.clamp((x.unsqueeze(-1) - self.lo) / (self.hi - self.lo + 1e-8), 0, 1)
		return x + (a * self.w.unsqueeze(0)).sum(-1)

class PeriodicEncoder(nn.Module):
	def __init__(self, n, nf = 8):
		super().__init__()
		self.om = nn.Parameter(torch.randn(n, nf))
		self.ph = nn.Parameter(torch.randn(n, nf) * 0.1)
		self.c = nn.Parameter(torch.zeros(n, nf))
	def forward(self, x):
		return x + (torch.sin(x.unsqueeze(-1) * self.om.unsqueeze(0) + self.ph.unsqueeze(0)) * self.c.unsqueeze(0)).sum(-1)

class FourierEncoder(nn.Module):
	def __init__(self, n, nf = 8):
		super().__init__()
		self.register_buffer('freq', torch.arange(1, nf + 1, dtype = torch.float32) * torch.pi)
		self.a = nn.Parameter(torch.zeros(n, nf))
		self.b = nn.Parameter(torch.zeros(n, nf))
	def forward(self, x):
		s = x.unsqueeze(-1) * self.freq
		return x + (torch.sin(s) * self.a.unsqueeze(0) + torch.cos(s) * self.b.unsqueeze(0)).sum(-1)

class MagnitudeDirectionEncoder(nn.Module):
	def __init__(self, n):
		super().__init__()
		self.wp = nn.Parameter(torch.ones(n))
		self.wn = nn.Parameter(torch.ones(n))
		self.b = nn.Parameter(torch.zeros(n))
	def forward(self, x): return F.relu(x) * self.wp - F.relu(-x) * self.wn + self.b

def build_encoder(v, n):
	enc = {'identity': IdentityEncoder, 'linear': LinearEncoder, 'ple': PLEEncoder,
		'periodic': PeriodicEncoder, 'fourier': FourierEncoder, 'magnitude_dir': MagnitudeDirectionEncoder}
	return enc[v]() if v == 'identity' else enc[v](n)

### 5. Kelly Architecture

In [ ]:
class AttentionHead(nn.Module):
	def __init__(self, n, s):
		super().__init__()
		self.w = nn.Parameter(torch.randn(n, n) * s)
		self.v = nn.Parameter(torch.randn(n, n) * s)
		self.sc = 1.0 / np.sqrt(n)
	def forward(self, y):
		return F.softmax((y @ self.w @ y.t()) * self.sc, dim = -1) @ (y @ self.v)

class TransformerBlock(nn.Module):
	def __init__(self, n, h, ff, s):
		super().__init__()
		self.heads = nn.ModuleList([AttentionHead(n, s) for _ in range(h)])
		self.w1 = nn.Parameter(torch.randn(n, ff) * (1.0 / ff))
		self.b1 = nn.Parameter(torch.zeros(ff))
		self.w2 = nn.Parameter(torch.randn(ff, n) * s)
		self.b2 = nn.Parameter(torch.zeros(n))
	def forward(self, y):
		y = sum(h(y) for h in self.heads) + y
		return F.relu(y @ self.w1 + self.b1) @ self.w2 + self.b2 + y

class PortfolioTransformer(nn.Module):
	def __init__(self, n, nb, nh, ff, enc):
		super().__init__()
		self.enc = enc
		s = 1.0 / n
		self.blocks = nn.ModuleList([TransformerBlock(n, nh, ff, s) for _ in range(nb)])
		self.lam = nn.Parameter(torch.randn(n) * s)
	def forward(self, x):
		y = self.enc(x)
		for b in self.blocks: y = b(y)
		return y @ self.lam
	def msrr_loss(self, x, r): return (1.0 - self.forward(x) @ r) ** 2

print(f'd = {d}, params: {sum(p.numel() for p in PortfolioTransformer(d, n_blocks, n_heads, d_ff, IdentityEncoder()).parameters()):,}')

### 6. Expanding Window Training

Each year, the model retrains from scratch on all data up to that point, with the last 24 months used as validation for early stopping. Predictions cover the next 12 months out of sample.

In [ ]:
def to_gpu_dict(month_keys):
	return {eom: {
		'x': torch.tensor(all_months[eom]['x'], dtype = torch.float32, device = device),
		'r': torch.tensor(all_months[eom]['r'], dtype = torch.float32, device = device),
		'ids': all_months[eom]['ids'],
	} for eom in month_keys if eom in all_months}


@torch.no_grad()
def val_rank_corr(model, vg):
	model.eval()
	corrs = []
	for m in vg.values():
		w = model(m['x']).cpu().numpy()
		r = m['r'].cpu().numpy()
		if len(w) < 10: continue
		c, _ = spearmanr(w, r)
		if not np.isnan(c): corrs.append(c)
	model.train()
	return float(np.mean(corrs)) if corrs else 0.0


def train_one_window(variant, train_gpu, val_gpu, seed):
	torch.manual_seed(seed)
	np.random.seed(seed)
	model = PortfolioTransformer(d, n_blocks, n_heads, d_ff, build_encoder(variant, d).to(device)).to(device)
	opt = torch.optim.Adam(model.parameters(), lr = lr, weight_decay = weight_decay)
	keys = list(train_gpu.keys())
	nm = len(keys)

	bv = -np.inf
	bs = None
	wait = 0

	for ep in range(1, n_epochs + 1):
		model.train()
		for idx in np.random.permutation(nm):
			opt.zero_grad()
			loss = model.msrr_loss(train_gpu[keys[idx]]['x'], train_gpu[keys[idx]]['r'])
			loss.backward()
			nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
			opt.step()
		vc = val_rank_corr(model, val_gpu)
		if vc > bv:
			bv = vc
			bs = {k: v.cpu().clone() for k, v in model.state_dict().items()}
			wait = 0
		else: wait += 1
		if wait >= patience: break

	if bs: model.load_state_dict(bs); model = model.to(device)
	return model, bv


def train_variant(variant):
	print(f'\nVariant: {variant}')
	vdir = results_dir / variant
	vdir.mkdir(parents = True, exist_ok = True)
	t0 = time.time()

	oos_predictions = {}
	year_log = []

	for oos_year in oos_years:
		# Split dates
		train_dates = [d for d in sorted_dates if d.year < oos_year]
		if len(train_dates) < val_size_months + 12: continue

		val_dates = train_dates[-val_size_months:]
		train_only = train_dates[:-val_size_months]
		oos_dates = [d for d in sorted_dates if d.year == oos_year]

		if len(train_only) < 12 or len(oos_dates) == 0: continue

		# Move to GPU
		tg = to_gpu_dict(train_only)
		vg = to_gpu_dict(val_dates)
		og = to_gpu_dict(oos_dates)

		# Train seeds and predict
		seed_preds = {eom: [] for eom in oos_dates if eom in og}
		best_vcs = []

		for seed in range(n_seeds):
			model, bvc = train_one_window(variant, tg, vg, seed)
			best_vcs.append(bvc)
			model.eval()
			with torch.no_grad():
				for eom in seed_preds:
					w = model(og[eom]['x']).cpu().numpy()
					seed_preds[eom].append(w)
			del model

		# Average predictions across seeds
		for eom in seed_preds:
			if len(seed_preds[eom]) == 0: continue
			w_sum = np.zeros(len(seed_preds[eom][0]))
			nv = 0
			for w in seed_preds[eom]:
				w64 = w.astype(np.float64)
				a = np.abs(w64).sum()
				if a > 1e-10: w_sum += w64 / a; nv += 1
			if nv > 0:
				oos_predictions[eom] = {
					'w': (w_sum / nv).astype(np.float32),
					'ids': all_months[eom]['ids'],
					'r': all_months[eom]['r'],
				}

		mean_vc = float(np.mean(best_vcs))
		year_log.append({'year': oos_year, 'train_months': len(train_only),
			'val_months': len(val_dates), 'oos_months': len(oos_dates), 'val_corr': mean_vc})

		elapsed = (time.time() - t0) / 60
		print(f'  {oos_year}: train {len(train_only)}m, val {len(val_dates)}m, '
			f'oos {len(oos_dates)}m, val corr {mean_vc:.4f}, {elapsed:.1f} min')

		del tg, vg, og
		gc.collect()
		torch.cuda.empty_cache()

	elapsed = (time.time() - t0) / 60
	print(f'  done: {len(oos_predictions)} OOS months in {elapsed:.1f} min')

	with open(vdir / f'{variant}_{country}_oos_predictions.pkl', 'wb') as f:
		pickle.dump(oos_predictions, f)
	with open(vdir / f'{variant}_{country}_year_log.json', 'w') as f:
		json.dump(year_log, f, indent = 2, default = float)

	return oos_predictions, year_log, elapsed

### 7. Evaluation

In [ ]:
def oos_rank_corr(preds):
	corrs = []
	for eom in sorted(preds.keys()):
		if len(preds[eom]['w']) < 10: continue
		c, _ = spearmanr(preds[eom]['w'], preds[eom]['r'])
		if not np.isnan(c): corrs.append(c)
	return float(np.mean(corrs)) if corrs else 0.0, corrs


def quintile_sim(preds, rf = 6, tc = 25):
	keys = sorted(preds.keys())
	if not keys: return np.array([]), []
	rset = set(keys[::rf])
	ml = []
	li = set()
	si = set()
	pl = set()
	ps = set()
	hl = []
	for eom in keys:
		m = preds[eom]
		w = m['w']
		r = m['r']
		ids = m['ids']
		tcv = 0.0
		if eom in rset:
			nq = max(1, int(len(w) * 0.20))
			so = np.argsort(w)
			li = set(ids[so[::-1][:nq]].tolist())
			si = set(ids[so[:nq]].tolist())
			to = (len(li - pl) + len(pl - li) + len(si - ps) + len(ps - si)) / max(nq, 1)
			tcv = to * tc / 10000.0
			pl = li
			ps = si
			hl.append({'eom': str(eom),
				'long': [{'id': i, 'gvkey': id_to_gvkey.get(i, '')} for i in sorted(li)],
				'short': [{'id': i, 'gvkey': id_to_gvkey.get(i, '')} for i in sorted(si)]})
		if not li: continue
		il = ids.tolist()
		lr = r[np.array([i in li for i in il])]
		sr = r[np.array([i in si for i in il])]
		ml.append((float(lr.mean()) if len(lr) > 0 else 0) - (float(sr.mean()) if len(sr) > 0 else 0) - tcv)
	return np.array(ml), hl


def score_weighted_sim(preds):
	ml = []
	for eom in sorted(preds.keys()):
		w = preds[eom]['w'].astype(np.float64)
		r = preds[eom]['r'].astype(np.float64)
		ww = w - w.mean()
		a = np.abs(ww).sum()
		if a > 1e-10: ml.append(float((ww / a) @ r))
	return np.array(ml)


def portfolio_metrics(rets, ppy = 12):
	if len(rets) == 0: return {}
	tw = float((1 + rets).prod())
	ann_ret = -1.0 if tw <= 0 else float(tw ** (ppy / len(rets)) - 1)
	av = float(rets.std() * np.sqrt(ppy))
	sr = ann_ret / max(av, 1e-8)
	se = float(np.sqrt((1 + 0.5 * sr ** 2) / len(rets)))
	pk = np.maximum.accumulate(np.cumprod(1 + rets))
	dd = float(((pk - np.cumprod(1 + rets)) / pk).max()) if len(pk) > 0 else 0
	return {'ann_ret': ann_ret, 'ann_vol': av, 'sharpe': sr, 'se_sharpe': se, 'max_dd': dd, 'n_months': len(rets)}


def evaluate_variant(preds, vname, vdir, year_log):
	print(f'\n  {vname} ({len(preds)} OOS months)')

	rc, rc_monthly = oos_rank_corr(preds)
	print(f'  Rank corr: {rc:.4f}')

	qr, holdings = quintile_sim(preds)
	qm = portfolio_metrics(qr)
	print(f'  Quintile   sharpe: {qm.get("sharpe", 0):.4f} (se {qm.get("se_sharpe", 0):.4f})  ret: {qm.get("ann_ret", 0) * 100:.2f}%')

	swr = score_weighted_sim(preds)
	swm = portfolio_metrics(swr)
	print(f'  Score wt   sharpe: {swm.get("sharpe", 0):.4f} (se {swm.get("se_sharpe", 0):.4f})  ret: {swm.get("ann_ret", 0) * 100:.2f}%')

	np.save(vdir / f'{vname}_{country}_quintile.npy', qr)
	np.save(vdir / f'{vname}_{country}_scorewt.npy', swr)
	with open(vdir / f'{vname}_{country}_holdings.json', 'w') as f:
		json.dump(holdings, f, indent = 2, default = str)
	with open(vdir / f'{vname}_{country}_rank_corrs.json', 'w') as f:
		json.dump({'mean': rc, 'monthly': rc_monthly}, f, default = float)

	return {'variant': vname, 'rank_corr': rc, 'quintile': qm, 'score_weighted': swm, 'year_log': year_log}

### 8. Per Variant Training

In [ ]:
results = {}

In [ ]:
preds_identity, ylog_identity, time_identity = train_variant('identity')
results['identity'] = evaluate_variant(preds_identity, 'identity', results_dir / 'identity', ylog_identity)
results['identity']['time_min'] = time_identity / 60

In [ ]:
preds_linear, ylog_linear, time_linear = train_variant('linear')
results['linear'] = evaluate_variant(preds_linear, 'linear', results_dir / 'linear', ylog_linear)
results['linear']['time_min'] = time_linear / 60

In [ ]:
preds_ple, ylog_ple, time_ple = train_variant('ple')
results['ple'] = evaluate_variant(preds_ple, 'ple', results_dir / 'ple', ylog_ple)
results['ple']['time_min'] = time_ple / 60

In [ ]:
preds_periodic, ylog_periodic, time_periodic = train_variant('periodic')
results['periodic'] = evaluate_variant(preds_periodic, 'periodic', results_dir / 'periodic', ylog_periodic)
results['periodic']['time_min'] = time_periodic / 60

In [ ]:
preds_fourier, ylog_fourier, time_fourier = train_variant('fourier')
results['fourier'] = evaluate_variant(preds_fourier, 'fourier', results_dir / 'fourier', ylog_fourier)
results['fourier']['time_min'] = time_fourier / 60

In [ ]:
preds_magnitude_dir, ylog_magnitude_dir, time_magnitude_dir = train_variant('magnitude_dir')
results['magnitude_dir'] = evaluate_variant(preds_magnitude_dir, 'magnitude_dir', results_dir / 'magnitude_dir', ylog_magnitude_dir)
results['magnitude_dir']['time_min'] = time_magnitude_dir / 60

### 9. Results

In [ ]:
print(f'Expanding Window Comparison: {country}')
print(f'{initial_train_years}yr initial, {val_size_months}m val, annual retrain, {n_seeds} seeds')
print()
print(f'{"Variant":<18} {"Corr":>6} {"Q":>7} {"SW":>7} {"Ret":>7} {"Time":>6}')
print()
for v, r in sorted(results.items(), key = lambda x: -x[1]['rank_corr']):
	q = r.get('quintile', {})
	sw = r.get('score_weighted', {})
	print(f'{v:<18} {r["rank_corr"]:6.4f} {q.get("sharpe", 0):7.3f} '
		f'{sw.get("sharpe", 0):7.3f} '
		f'{q.get("ann_ret", 0) * 100:6.2f}% {r.get("time_min", 0):5.0f}m')

with open(results_dir / f'{country}_expanding_comparison.json', 'w') as f:
	json.dump(results, f, indent = 2, default = lambda x: float(x) if hasattr(x, '__float__') else str(x))
print(f'\nSaved: {results_dir / f"{country}_expanding_comparison.json"}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize = (14, 10))
fig.suptitle(f'{country}: Expanding Window ({initial_train_years}yr initial)', fontsize = 14)
vs = list(results.keys())
lb = [v.replace('_', ' ').title() for v in vs]
x = np.arange(len(vs))

axes[0,0].bar(x, [results[v]['rank_corr'] for v in vs])
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(lb, rotation = 25, ha = 'right')
axes[0,0].set_title('OOS Rank Correlation')
axes[0,0].grid(axis = 'y', alpha = 0.3)

for i, (key, title) in enumerate([('quintile', 'Quintile Sharpe'), ('score_weighted', 'Score Weighted')]):
	sh = [results[v].get(key, {}).get('sharpe', 0) for v in vs]
	se = [results[v].get(key, {}).get('se_sharpe', 0) for v in vs]
	axes[0, i + 1].bar(x, sh, yerr = se, capsize = 3)
	axes[0, i + 1].set_xticks(x)
	axes[0, i + 1].set_xticklabels(lb, rotation = 25, ha = 'right')
	axes[0, i + 1].set_title(title)
	axes[0, i + 1].grid(axis = 'y', alpha = 0.3)

# Val corr over time (expanding window)
for v in vs:
	yl = results[v].get('year_log', [])
	if yl:
		axes[1, 0].plot([y['year'] for y in yl], [y['val_corr'] for y in yl],
			marker = 'o', markersize = 3, label = v.replace('_', ' ').title())
axes[1, 0].set_title('Val Rank Corr by Retrain Year')
axes[1, 0].legend(fontsize = 7)
axes[1, 0].grid(alpha = 0.3)

# Cumulative wealth
for v in vs:
	p = results_dir / v / f'{v}_{country}_quintile.npy'
	if p.exists():
		axes[1, 1].plot(np.cumprod(1 + np.load(p)), label = v.replace('_', ' ').title())
axes[1, 1].set_title('Cumulative Wealth (Quintile)')
axes[1, 1].legend(fontsize = 7)
axes[1, 1].grid(alpha = 0.3)

plt.tight_layout()
plt.savefig(results_dir / f'{country}_expanding_comparison.png', dpi = 150, bbox_inches = 'tight')
plt.show()